In [2]:
import yfinance as yf
import pandas as pd

# Lista de tickers
tickersus = ['^GSPC','USDBRL=X','AAPL','AIG','BAC','RIO','DHI','EXC','KMB',
             'KO','LOPE','LYB','MGA','MSFT','MSTR','NUE','NVDA','TGT',
             'TMUS','UPS','UNH','XOM']

# Dicionário solicitado para armazenar os dados (cada valor é um dict por ticker)
yfstockusdata = {}

# Conjunto para acumular todas as chaves encontradas em todos os tickers
all_keys = set()

for t in tickersus:
    try:
        tk = yf.Ticker(t)
        info = tk.info or {}
        # Garante que o ticker esteja presente como campo
        info_row = {'Ticker': t}
        # adiciona todas as chaves/valores retornados por .info
        for k, v in info.items():
            info_row[k] = v
            all_keys.add(k)
        # armazena a linha
        yfstockusdata[t] = info_row
    except Exception as e:
        # Em caso de falha, registra o ticker com apenas o campo 'Ticker'
        # e mantém as demais chaves ausentes (serão NaN no DataFrame)
        yfstockusdata[t] = {'Ticker': t}
        # (opcional) você pode logar o erro se quiser:
        # print(f"Erro em {t}: {e}")

# Para garantir colunas ordenadas com 'Ticker' primeiro, montamos explicitamente a lista de colunas
cols_other = sorted(all_keys - {'Ticker'})  # ordena alfabeticamente as demais chaves (opcional)
cols_final = ['Ticker'] + cols_other

# Converte o dicionário para DataFrame mantendo índice numérico (0,1,2,...)
# Note: usamos list(yfstockusdata.values()) para preservar cada dict como uma linha
yfstockus = pd.DataFrame(list(yfstockusdata.values()))

# Reindexa as colunas para garantir 'Ticker' como primeira e as restantes presentes
# Existem casos em que algumas chaves não aparecem em nenhum ticker — isso não ocorrerá, 
# pois all_keys foi montado a partir de .info de cada ticker.
# Ainda assim, para segurança:
existing_cols = [c for c in cols_final if c in yfstockus.columns]
yfstockus = yfstockus[existing_cols + [c for c in yfstockus.columns if c not in existing_cols]]

# Exibe o DataFrame (uso em Jupyter/IPython)
display(yfstockus)


,Ticker,52WeekChange,SandP52WeekChange,address1,address2,allTimeHigh,allTimeLow,ask,askSize,auditRisk,...,trailingPE,trailingPegRatio,triggerable,twoHundredDayAverage,twoHundredDayAverageChange,twoHundredDayAverageChangePercent,typeDisp,volume,website,zip
0,^GSPC,19.736935,NaN,NaN,NaN,6920.340000,4.400000,6881.6700,0,NaN,...,NaN,NaN,True,6111.516000,728.684100,0.119231,Index,3777566000,NaN,NaN
1,USDBRL=X,NaN,NaN,NaN,NaN,6.411100,1.525200,5.3772,0,NaN,...,NaN,NaN,True,5.589596,-0.214396,-0.038356,Currency,0,NaN,NaN
2,AAPL,0.217828,0.197369,One Apple Park Way,NaN,277.320000,0.049107,285.5800,1,7.0,...,36.242626,2.4662,True,223.302000,47.067993,0.210782,Equity,86167123,https://www.apple.com,95014
3,AIG,0.034456,0.197369,1271 Avenue of the Americas,NaN,2075.000000,6.600000,78.9200,3,6.0,...,14.758879,NaN,True,80.664650,-1.704651,-0.021133,Equity,2338560,https://www.aig.com,10020
4,BAC,0.293249,0.197369,Bank of America Corporate Center,100 North Tryon Street,55.080000,0.828125,53.5000,47,6.0,...,14.603825,1.2606,True,45.860750,7.589252,0.165485,Equity,39278767,https://www.bankofamerica.com,28255
5,RIO,0.103523,0.197369,6 St James’s Square,NaN,139.662500,7.500000,72.5300,2,NaN,...,11.423566,NaN,True,61.810000,9.929996,0.160654,Equity,7777899,https://www.riotinto.com,SW1Y 4AD
6,DHI,-0.126860,0.197369,1341 Horton Circle,NaN,199.850000,1.051366,149.2900,2,8.0,...,12.885048,1.8235,True,140.596800,8.483200,0.060337,Equity,2991760,https://www.drhorton.com,76011
7,EXC,0.214643,0.197369,10 South Dearborn Street,54th Floor PO Box 805398,65.713264,3.209700,48.4400,1,3.0,...,17.536121,2.0601,True,44.104300,2.015697,0.045703,Equity,8716383,https://www.exeloncorp.com,60680-5379
8,KMB,-0.104905,0.197369,PO Box 619100,NaN,160.160000,2.262105,119.7900,1,3.0,...,20.255499,4.8401,True,131.944660,-12.234657,-0.092726,Equity,3832896,https://www.kimberly-clark.com,75261-9100
9,KO,0.058209,0.197369,One Coca-Cola Plaza,NaN,74.380000,0.182292,68.8900,23,2.0,...,22.814570,2.2073,True,69.205150,-0.305145,-0.004409,Equity,11611988,https://www.coca-colacompany.com,30313


In [4]:
# --- Função de sanitização por valor ---
def sanitize_value(v, maxlen=500):
    # None / NaN
    if v is None:
        return ''
    # floats: rejeitar NaN/Inf
    if isinstance(v, float):
        if not np.isfinite(v):
            return ''
        return float(v)
    # números inteiros e booleanos são OK
    if isinstance(v, (int, bool, np.integer, np.bool_)):
        return int(v) if isinstance(v, (int, np.integer)) else bool(v)
    # strings curtas OK (trunca se muito longas)
    if isinstance(v, str):
        return v if len(v) <= maxlen else v[:maxlen]
    # séries temporais / numpy types convertidos para string truncada
    # para todos os demais tipos (dict, list, Timestamp, ndarray, Decimal, etc.)
    try:
        s = str(v)
        return s if len(s) <= maxlen else s[:maxlen]
    except Exception:
        return ''

# --- Sanitizar coluna a coluna usando Series.map (evita applymap) ---
yfstockus_sanitized = yfstockus.copy()

for col in yfstockus_sanitized.columns:
    # Usa map para aplicar a função de maneira vetorizada por coluna
    yfstockus_sanitized[col] = yfstockus_sanitized[col].map(lambda x: sanitize_value(x))

# --- Monta as linhas para enviar ao Google Sheets ---
rows = [yfstockus_sanitized.columns.tolist()] + yfstockus_sanitized.values.tolist()

# --- Abre/Cria worksheet e atualiza ---
try:
    wssyfstockus = wb.worksheet('YFStockUS')
except Exception as e:
    # se não existir, cria (ajuste linhas/cols se quiser um tamanho específico)
    wssyfstockus = wb.add_worksheet(title='YFStockUS', rows=str(len(rows)+10), cols=str(len(rows[0])+5))

# Faz o update (agora com dados serializáveis)
wssyfstockus.update(rows)


NameError: name 'np' is not defined